In [1]:
!pip install transformers datasets accelerate scikit-learn -q

In [2]:
import torch
import numpy as np
import pandas as pd
import re
import matplotlib.pyplot as plt
import warnings
from collections import Counter
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [3]:

print("Loading IMDB dataset...")
full_dataset = load_dataset("imdb")


def clean_text(example):
    text = example['text']
    if text is None: return {'text': ""}

    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.lower().strip()
    example['text'] = text
    return example



print("Cleaning and handling missing values...")
cleaned_dataset = full_dataset.filter(lambda x: x['text'] is not None and len(x['text']) > 0)
cleaned_dataset = cleaned_dataset.map(clean_text)

TRAIN_SUBSET_SIZE = 2000
TEST_SUBSET_SIZE = 500

train_data = cleaned_dataset['train'].shuffle(seed=42).select(range(TRAIN_SUBSET_SIZE))
test_data = cleaned_dataset['test'].shuffle(seed=42).select(range(TEST_SUBSET_SIZE))

print(f"Subset created: {len(train_data)} train samples, {len(test_data)} test samples.")

Loading IMDB dataset...


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

...

Filter:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Subset created: 2000 train samples, 500 test samples.


In [4]:

train_val_split = train_data.train_test_split(test_size=0.2, seed=42)

train_dataset = train_val_split['train']
val_dataset = train_val_split['test']
test_dataset = test_data

print(f"Final splits -> Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")
print(f"\nSample distribution: {Counter(train_dataset['label'])}")

Final splits -> Train: 1600, Val: 400, Test: 500

Sample distribution: Counter({1: 812, 0: 788})


In [5]:
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=256
    )

print("Tokenizing datasets...")
train_tokenized = train_dataset.map(tokenize_fn, batched=True)
val_tokenized = val_dataset.map(tokenize_fn, batched=True)
test_tokenized = test_dataset.map(tokenize_fn, batched=True)

train_tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
val_tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

print("Tokenization complete.")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizing datasets...


Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenization complete.


In [6]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    acc = accuracy_score(labels, predictions)
    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

In [ ]:
print("\n--- EXPERIMENT 1: Frozen BERT ---")
model_exp1 = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

for param in model_exp1.bert.parameters():
    param.requires_grad = False

training_args_exp1 = TrainingArguments(
    output_dir='./results_exp1',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
    logging_steps=50
)

trainer_exp1 = Trainer(
    model=model_exp1,
    args=training_args_exp1,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    compute_metrics=compute_metrics
)

trainer_exp1.train()
results_exp1 = trainer_exp1.evaluate(test_tokenized)
print(f"Experiment 1 Test Accuracy: {results_exp1['eval_accuracy']:.4f}")


--- EXPERIMENT 1: Frozen BERT ---


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.693987,0.684149,0.585000,0.553922,0.601064,0.576531
2,0.690951,0.684969,0.552500,0.516129,0.765957,0.616702


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.693987,0.684149,0.585000,0.553922,0.601064,0.576531
2,0.690951,0.684969,0.552500,0.516129,0.765957,0.616702
3,0.687552,0.684437,0.560000,0.521898,0.760638,0.619048


In [ ]:
print("\n--- EXPERIMENT 2: Fine-tune Last 2 Layers ---")
model_exp2 = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

for param in model_exp2.bert.parameters():
    param.requires_grad = False

for layer in model_exp2.bert.encoder.layer[-2:]:
     for param in layer.parameters():
        param.requires_grad = True
for param in model_exp2.bert.pooler.parameters():
    param.requires_grad = True

training_args_exp2 = TrainingArguments(
    output_dir='./results_exp2',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
    lr_scheduler_type="linear",
    logging_steps=50
)

trainer_exp2 = Trainer(
    model=model_exp2,
    args=training_args_exp2,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

trainer_exp2.train()
results_exp2 = trainer_exp2.evaluate(test_tokenized)
print(f"Experiment 2 Test Accuracy: {results_exp2['eval_accuracy']:.4f}")


--- EXPERIMENT 2: Fine-tune Last 2 Layers ---


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.582611,0.468241,0.802500,0.842767,0.712766,0.772334
2,0.383597,0.310719,0.870000,0.869565,0.851064,0.860215
3,0.316937,0.298169,0.870000,0.846939,0.882979,0.864583


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

In [4]:
metrics_list = ['accuracy', 'precision', 'recall', 'f1']

data = {
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Exp 1 (Frozen)': [results_exp1[f'eval_{m}'] for m in metrics_list],
    'Exp 2 (Last 2)': [results_exp2[f'eval_{m}'] for m in metrics_list],
    'Bonus (DistilBERT)': [results_distil[f'eval_{m}'] for m in metrics_list]
}

comparison_df = pd.DataFrame(data)
print("\n--- MODEL COMPARISON ---")
print(comparison_df.to_string(index=False))

comparison_df.set_index('Metric').plot(kind='bar', figsize=(10, 6))
plt.title('Performance Comparison across Experiments')
plt.ylim(0, 1.0)
plt.xticks(rotation=45)
plt.legend(loc='lower right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

NameError: name 'results_exp1' is not defined

In [2]:
def plot_cm(trainer, dataset, title, ax):
    preds = trainer.predict(dataset)
    y_true = preds.label_ids
    y_pred = np.argmax(preds.predictions, axis=-1)
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Neg', 'Pos'])
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(title)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plot_cm(trainer_exp1, test_tokenized, "Exp 1: Frozen BERT", axes[0])
plot_cm(trainer_exp2, test_tokenized, "Exp 2: Last 2 Layers", axes[1])
plot_cm(trainer_distil, test_distil, "Bonus: DistilBERT", axes[2])
plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined